# 🤖 FinAgent – AI Analysis (Groq LLaMA 3.3 - 11 Assets)
Chạy **Cell 1** cài thư viện, rồi chạy **Cell 2** là xong!

In [1]:
# CELL 1 — Cài thư viện (chỉ cần chạy 1 lần)
import subprocess
subprocess.run(["pip", "install", "groq", "openpyxl"])
print('✅ Cài xong!')

✅ Cài xong!


In [2]:
# ============================================================
# CELL 2 — Toàn bộ phân tích AI cho 11 assets (~30-60 giây)
# Chỉ gọi API 4 lần duy nhất → nhanh, không tốn quota
# ============================================================
import json
from datetime import datetime
import pandas as pd
from groq import Groq

# ✏️ CHỈ SỬA 2 DÒNG NÀY
GROQ_API_KEY = "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"    # ← dán key Groq thật vào
INPUT_FILE   = "/Users/nguyenhoangnguyetanh/Downloads/hihi.xlsx"   # ← tên file Excel

# ── Setup ────────────────────────────────────────────────────
client  = Groq(api_key=GROQ_API_KEY)
TICKERS = ["AAPL","AMZN","JNJ","JPM","META","MSFT","NVDA","TSLA","SP500","Gold","Oil"]
COMPARE_PAIRS = [("NVDA","TSLA"),("AAPL","MSFT"),("Gold","SP500")]
SYSTEM  = """You are a quantitative financial analyst assistant.
Be concise, professional, under 150 words per section.
Always reference specific numbers from the data provided.
Do NOT invent figures not in the data."""

# ── Load data ────────────────────────────────────────────────
df = pd.read_excel(INPUT_FILE, parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)
print(f"✅ Loaded: {df.shape[0]} rows × {df.shape[1]} cols")

# ── Helper: build 30-day snapshot ────────────────────────────
def build_snapshot(ticker):
    close_col = f"{ticker}_Close"
    if close_col not in df.columns: return None
    cols = ["Date", close_col] + [
        f"{ticker}{s}" for s in ["_return","_MA7","_MA30","_volatility"]
        if f"{ticker}{s}" in df.columns
    ]
    recent = df[cols].dropna(subset=[close_col]).tail(30)
    if recent.empty: return None
    last, first = recent.iloc[-1], recent.iloc[0]
    return {
        "ticker":           ticker,
        "period":           f"{first['Date'].date()} to {last['Date'].date()}",
        "latest_close":     round(float(last[close_col]), 4),
        "30d_return_pct":   round((float(last[close_col])/float(first[close_col])-1)*100, 2),
        "latest_ma7":       round(float(last[f"{ticker}_MA7"]), 4)  if f"{ticker}_MA7"        in recent.columns else None,
        "latest_ma30":      round(float(last[f"{ticker}_MA30"]), 4) if f"{ticker}_MA30"       in recent.columns else None,
        "avg_volatility":   round(float(recent[f"{ticker}_volatility"].mean())*100, 4) if f"{ticker}_volatility" in recent.columns else None,
        "min_daily_return": round(float(recent[f"{ticker}_return"].min())*100, 4)      if f"{ticker}_return"    in recent.columns else None,
        "max_daily_return": round(float(recent[f"{ticker}_return"].max())*100, 4)      if f"{ticker}_return"    in recent.columns else None,
    }

# ── Helper: detect anomalies ─────────────────────────────────
def detect_anomalies():
    out = []
    for t in TICKERS:
        col = f"{t}_return"
        if col not in df.columns: continue
        s = df[col].dropna()
        mean, std = s.mean(), s.std()
        if std == 0: continue
        flagged = df[["Date",col]].dropna()
        flagged = flagged[(abs(flagged[col]-mean)/std) >= 3]
        for _, row in flagged.iterrows():
            out.append({"ticker":t, "date":str(row["Date"].date()),
                        "return_pct":round(float(row[col])*100,4),
                        "z_score":round((float(row[col])-mean)/std,2)})
    out.sort(key=lambda x: abs(x["z_score"]), reverse=True)
    return out[:15]

# ── Helper: call Groq ─────────────────────────────────────────
def ask_groq(prompt):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role":"system", "content":SYSTEM},
            {"role":"user",   "content":prompt}
        ],
        max_tokens=1024,
        temperature=0.3
    )
    return response.choices[0].message.content.strip()

# ── Build snapshots ───────────────────────────────────────────
snapshots = []
for t in TICKERS:
    s = build_snapshot(t)
    if s: snapshots.append(s)
    else: print(f"  ⚠ Bỏ qua {t}: không tìm thấy cột")

results = {
    "generated_at":      datetime.now().isoformat(timespec="seconds"),
    "trend_summaries":   {},
    "anomaly_commentary":"",
    "risk_commentary":   "",
    "comparisons":       {}
}

# ── [1/4] Trend summaries — 1 lần gọi cho tất cả ─────────────
print("\n[1/4] Trend summaries...", end=" ", flush=True)
raw = ask_groq(
    f"For each of the {len(snapshots)} assets below, write a 2-3 sentence trend summary "
    f"referencing MA7 vs MA30 relationship and the 30-day return. "
    f"Respond ONLY as valid JSON with ticker as key and summary as value, no extra text.\n\n"
    f"Example format: {{\"AAPL\": \"summary here\", \"NVDA\": \"summary here\"}}\n\n"
    f"Data:\n{json.dumps(snapshots, indent=2)}"
)
try:
    clean = raw[raw.index("{"):raw.rindex("}")+1]
    results["trend_summaries"] = json.loads(clean)
    print("✅")
except:
    results["trend_summaries"] = {s["ticker"]: "See raw output" for s in snapshots}
    print("⚠ JSON parse issue, saved raw")

# ── [2/4] Anomaly commentary — 1 lần gọi ─────────────────────
print("[2/4] Anomaly detection...", end=" ", flush=True)
anomalies = detect_anomalies()
results["anomaly_commentary"] = ask_groq(
    f"Write a 4-6 sentence paragraph about the most significant return anomalies across all assets "
    f"(returns >= 3 standard deviations from mean). Mention specific tickers, dates, and z-scores.\n\n"
    f"Anomaly data:\n{json.dumps(anomalies, indent=2)}"
) if anomalies else "No significant anomalies detected across all assets."
print(f"✅ ({len(anomalies)} anomalies)")

# ── [3/4] Risk commentary — 1 lần gọi ────────────────────────
print("[3/4] Risk commentary...", end=" ", flush=True)
risk_data = [
    {"ticker":s["ticker"], "avg_volatility%":s["avg_volatility"],
     "worst_day%":s["min_daily_return"], "best_day%":s["max_daily_return"]}
    for s in snapshots
]
results["risk_commentary"] = ask_groq(
    f"Rank all {len(risk_data)} assets from highest to lowest risk. "
    f"Write a 5-6 sentence risk commentary referencing specific volatility percentages.\n\n"
    f"Risk data:\n{json.dumps(risk_data, indent=2)}"
)
print("✅")

# ── [4/4] Comparisons — 1 lần gọi cho tất cả pairs ───────────
print("[4/4] Comparisons...", end=" ", flush=True)
pairs_data = []
for (a, b) in COMPARE_PAIRS:
    sa = next((s for s in snapshots if s["ticker"]==a), None)
    sb = next((s for s in snapshots if s["ticker"]==b), None)
    if sa and sb:
        pairs_data.append({"pair": f"{a} vs {b}", "A": sa, "B": sb})

raw_pairs = ask_groq(
    f"For each pair below, write a 3-4 sentence comparison covering return performance, "
    f"volatility difference, trend direction, and a risk-adjusted observation. "
    f"Respond ONLY as valid JSON with pair name as key, no extra text.\n\n"
    f"Example: {{\"NVDA vs TSLA\": \"comparison here\", \"AAPL vs MSFT\": \"comparison here\"}}\n\n"
    f"Data:\n{json.dumps(pairs_data, indent=2)}"
)
try:
    clean = raw_pairs[raw_pairs.index("{"):raw_pairs.rindex("}")+1]
    results["comparisons"] = json.loads(clean)
    print("✅")
except:
    results["comparisons"] = {f"{a} vs {b}": raw_pairs for a,b in COMPARE_PAIRS}
    print("⚠ JSON parse issue, saved raw")

# ── Print full report ─────────────────────────────────────────
sep = "=" * 65
print(f"\n\n{sep}")
print("  FINAGENT – AI ANALYSIS REPORT (Groq LLaMA 3.3 70B)")
print(f"  Generated : {results['generated_at']}")
print(f"  Assets    : {', '.join(TICKERS)}")
print(sep)

print("\n📈  SECTION 1 — TREND SUMMARIES")
print("-" * 65)
for ticker, text in results["trend_summaries"].items():
    print(f"\n▶ {ticker}")
    print(text)

print("\n\n⚠   SECTION 2 — ANOMALY COMMENTARY")
print("-" * 65)
print(results["anomaly_commentary"])

print("\n\n🔴  SECTION 3 — RISK COMMENTARY")
print("-" * 65)
print(results["risk_commentary"])

print("\n\n⚖   SECTION 4 — ASSET COMPARISONS")
print("-" * 65)
for label, text in results["comparisons"].items():
    print(f"\n▶ {label}")
    print(text)

print(f"\n{sep}")

# ── Save JSON ─────────────────────────────────────────────────
with open("ai_analysis_output.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("\n💾 Saved: ai_analysis_output.json")

✅ Loaded: 472 rows × 89 cols

[1/4] Trend summaries... ✅
[2/4] Anomaly detection... ✅ (15 anomalies)
[3/4] Risk commentary... ✅
[4/4] Comparisons... ✅


  FINAGENT – AI ANALYSIS REPORT (Groq LLaMA 3.3 70B)
  Generated : 2026-05-19T08:16:20
  Assets    : AAPL, AMZN, JNJ, JPM, META, MSFT, NVDA, TSLA, SP500, Gold, Oil

📈  SECTION 1 — TREND SUMMARIES
-----------------------------------------------------------------

▶ AAPL
AAPL's MA7 is above its MA30, with a 30-day return of 9.82%. This indicates a strong upward trend. The latest MA7 is 253.4081 and MA30 is 242.4397.

▶ AMZN
AMZN's MA7 is above its MA30, with a 30-day return of 8.77%. This suggests a positive trend. The latest MA7 is 224.36 and MA30 is 217.7917.

▶ JNJ
JNJ's MA7 is below its MA30, with a 30-day return of -5.81%. This indicates a downward trend. The latest MA7 is 139.8362 and MA30 is 143.8884.

▶ JPM
JPM's MA7 is below its MA30, with a 30-day return of -2.17%. This suggests a weak trend. The latest MA7 is 233.0053 and MA30